# Decision Trees — ID3, pruning, assumptions and validation
Example 1: ID3 on the commute table · Example 2: Breast Cancer Wisconsin (1 = malignant)

In [ ]:
# !pip install scikit-learn scipy pandas matplotlib

## Example 1 · ID3 (entropy and information gain)

In [ ]:
import numpy as np, pandas as pd

# Experience table (slide "Sample Experience Table")
df = pd.DataFrame({
  'Hour':     '8 8 10 9 9 10 10 9 9 10 10 8 9'.split(),
  'Weather':  ('Sunny Cloudy Sunny Rainy Sunny Sunny Cloudy '
               'Rainy Sunny Cloudy Rainy Cloudy Sunny').split(),
  'Accident': 'No No No Yes Yes No No No Yes Yes No Yes No'.split(),
  'Stall':    'No Yes No No Yes No No No No Yes No No No'.split(),
  'Commute':  ('Long Long Short Long Long Short Short '
               'Medium Long Long Short Long Medium').split()})

def entropy(y):
    p = y.value_counts(normalize=True)
    return -(p * np.log2(p)).sum()

def info_gain(df, a, target='Commute'):
    w = df[a].value_counts(normalize=True)
    exp_H = sum(w[v] * entropy(df.loc[df[a] == v, target])
                for v in w.index)
    return round(exp_H, 4), round(entropy(df[target]) - exp_H, 4)

print('H(S) =', round(entropy(df['Commute']), 4))
for a in ['Hour', 'Weather', 'Accident', 'Stall']:
    print(a, info_gain(df, a))     # (expected entropy, information gain)


## Example 2 · Complexity, pruning and tuning

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     GridSearchCV, validation_curve)
from sklearn.tree import DecisionTreeClassifier, plot_tree

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)        # 1 = malignant
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3,
                                      stratify=y, random_state=42)
cv = StratifiedKFold(10, shuffle=True, random_state=0)

# A3 · Bias–variance: validation curve (train vs CV accuracy)
depths = range(1, 16)
tr, va = validation_curve(DecisionTreeClassifier(random_state=0), Xtr, ytr,
                          param_name='max_depth', param_range=depths, cv=cv)
print('gap train-CV (full tree):', round(tr.mean(1)[-1] - va.mean(1)[-1], 3))

# Cost-complexity pruning path → candidate alphas
path = DecisionTreeClassifier(random_state=0).cost_complexity_pruning_path(Xtr, ytr)

# Tune depth, leaf size and alpha with CV (class_weight for imbalance)
grid = {'max_depth': [2, 3, 4, 5, 6, None], 'min_samples_leaf': [1, 5, 10, 20],
        'ccp_alpha': [0, 0.005, 0.01, 0.02]}
gs = GridSearchCV(DecisionTreeClassifier(random_state=0, class_weight='balanced'),
                  grid, cv=cv, scoring='roc_auc').fit(Xtr, ytr)
tree = gs.best_estimator_
print(gs.best_params_, round(gs.best_score_, 3))     # depth 4, leaf 20 · AUC 0.963
plot_tree(tree, feature_names=X.columns, filled=True, proportion=True)


## Example 2 · Diagnostics: shift (A1), stability (A4), importance (A5), comparison (A6)

In [ ]:
from scipy import stats
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# A1 · Same distribution in train/test: KS per feature + adversarial AUC ≈ 0.5
ks_p = [stats.ks_2samp(Xtr[c], Xte[c]).pvalue for c in X.columns]
Z, z = pd.concat([Xtr, Xte]), np.r_[np.zeros(len(Xtr)), np.ones(len(Xte))]
adv = cross_val_score(DecisionTreeClassifier(max_depth=4), Z, z, cv=5, scoring='roc_auc')

# A4 · Stability: root split across 200 bootstrap samples
rng, roots = np.random.default_rng(0), []
for b in range(200):
    i = rng.integers(0, len(Xtr), len(Xtr))
    m = DecisionTreeClassifier(random_state=b).fit(Xtr.iloc[i], ytr.iloc[i])
    roots.append(X.columns[m.tree_.feature[0]])
print(pd.Series(roots).value_counts(normalize=True).head(3))

# A5 · Importance: permutation on test set (MDI is biased)
pi = permutation_importance(tree, Xte, yte, n_repeats=30, random_state=0)

# Model comparison: McNemar test vs. logistic regression
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
ok_t, ok_l = tree.predict(Xte) == yte, lr.predict(Xte) == yte
b, c = (ok_t & ~ok_l).sum(), (~ok_t & ok_l).sum()
chi2 = (abs(b - c) - 1) ** 2 / (b + c)
print('McNemar chi2 =', round(chi2, 2), 'p =', round(stats.chi2.sf(chi2, 1), 3))


## Graphical diagnostics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
# A3: validation curve
ax[0].plot(depths, tr.mean(1), 'o-', label='train'); ax[0].plot(depths, va.mean(1), 'o-', label='CV')
ax[0].set_xlabel('max_depth'); ax[0].legend(); ax[0].set_title('A3 · validation curve')
# A5: permutation importance (top 8)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values().tail(8)
imp.plot.barh(ax=ax[1]); ax[1].set_title('A5 · permutation importance')
# A6: calibration
for m, name in [(tree, 'tree'), (lr, 'logistic')]:
    fy, fx = calibration_curve(yte, m.predict_proba(Xte)[:, 1], n_bins=6, strategy='quantile')
    ax[2].plot(fx, fy, 'o-', label=name)
ax[2].plot([0, 1], [0, 1], '--', c='gray'); ax[2].legend(); ax[2].set_title('A6 · calibration')
plt.tight_layout(); plt.show()